In [1]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta
import plotly.express as px
from IPython.display import HTML
from IPython.display import Image
import warnings
warnings.filterwarnings('ignore')

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-08-14 18:03:53 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/mmm_tdc/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



In [ ]:
sql = """
select finaltrxyear, -- Año
       finaltrxmonth, -- Mes
       finaltrxday, -- Día
       finaltrxhour, -- Hora
case 
   when (UPPER(TRIM(channel))='NEG' and UPPER(TRIM(devicenameid)) in ('APP','MOBILE')) then 'APP_NEG'
   when (UPPER(TRIM(channel))='APP' and UPPER(TRIM(devicenameid))='APP' ) then 'Mi Bancolombia'
   when (UPPER(TRIM(channel))='SVP' and UPPER(TRIM(devicenameid))='SVP') then 'SVP'
   when UPPER(TRIM(channel))='IGB' then  'APP_INVERSIONES'
   when UPPER(TRIM(channel))='SVI' then  'SUCURSAL_VIRTUAL_INVERSIONES'
   else
       upper(UPPER(TRIM(channel)))
       end as channel, transactioncode,
   case 
              when trim(field9) in ('Solicitud','Respuesta')  then concat('RCC-', transactioncodedesc,'-',field9)
              when trim(descriptionfunctions) in('Entrante Asíncrona','Resultado Asíncrona','Resultado Asíncrona','Resultado Síncrona') then concat('RCC-', transactioncodedesc,'-',descriptionfunctions)
              else concat('RCC-', transactioncodedesc)
            end as transactioncodedesc, 
            responsecode, responsecodedesc, transactionstate,
count(distinct cast(trim(REGEXP_REPLACE(documentnumber, '[^0-9]', '')) as bigint) ) as documentnumber, --clientes unicos
count(*) as cantidad -- Revisión de temas con documentos.
from s_canales.knsis_scanales_repocentprocedata
where year=2026 and finaltrxmonth = 8 and finaltrxday = 18
   and finaltrxyear=2026
   and ((UPPER(TRIM(channel))='APP' and UPPER(TRIM(devicenameid))='APP') 
   or  (UPPER(TRIM(channel))='SVP' and UPPER(TRIM(devicenameid))='SVP'))
   and cast(responsecode as int) <> 0  -- Se puede cambair a exitosas con = 0
   and cast(transactioncode as int) in(369) -- Transaccion de autenticación.
   group by 1,2,3,4,5,6,7,8,9,10 limit 10;
   group by 1,2,3,4,5,6,7,8,9,10 limit 10;
"""